In [11]:
import yfinance as yf
import pandas as pd

In [12]:
# Create a list of tickers
df = pd.read_csv("../data/processed/stocks_transformed_verified.csv")
tickers = df["ticker"].tolist()
tickers

['MP',
 'USAR',
 'LYC.F',
 'UUUU',
 'ALB',
 'SQM',
 '002460.KS',
 '3402.T',
 'HXL',
 '4188.T',
 'MMM',
 'DD',
 'BAS.DE',
 '1COV.F',
 'EVK.DE',
 'MT',
 'NVDA',
 'GOOGL',
 'META',
 '002230.KQ',
 'U',
 'GOOGL',
 'QCOM',
 'AMD',
 'INTC',
 'AMBA',
 'LSCC',
 'CEVA',
 'NXPI',
 'STM',
 'IFX.DE',
 'TXN',
 '2454.T',
 'MPWR',
 '300750.SZ',
 '373220.KS',
 '006400.KS',
 '6752.T',
 '002594.SZ',
 '300014.SZ',
 '6981.T',
 '6762.T',
 'NVTS',
 'WOLF',
 'ADI',
 'SONY',
 '005930.KS',
 'HSAI',
 'OUST',
 'LAZR',
 'NOVT',
 '2498.T',
 'ALGM',
 'VPG',
 'CGNX',
 '688322.SS',
 '603501.SS',
 '6324.T',
 '688017.SS',
 '6268.T',
 '6481.T',
 '601100.SS',
 '6594.T',
 'RRX',
 '603728.SS',
 '002979.SZ',
 '688698.SS',
 'ALNT',
 'RBC',
 'AME',
 'RRX',
 '002050.SZ',
 '601689.SS',
 '003021.SZ',
 '300124.SZ',
 'TSLA',
 '005380.KS',
 '688836.SS',
 '9880.T',
 '1810.T',
 'XPEV',
 'SIE.DE',
 'ABBN.SW',
 '000333.SZ',
 '6954.T',
 '6506.T',
 '2317.T',
 'ROK',
 'JBL',
 'AMZN',
 'BMW.DE',
 'MBG.DE',
 '005380.KS',
 'GXO',
 'DHL.DE',
 

In [13]:
# I create a list to store the KPI data for each ticker. I will create a DataFrame from this list at the end of the loop.
kpi_data = []

In [14]:
# Calculate KPIs for each stock.
for ticker in tickers:

    try:
        t = yf.Ticker(ticker)
        
        # KPI #1: PEG ratio.
        forward_EPS_growth = ((t.info['forwardEps'] - t.info['epsCurrentYear']) / t.info['epsCurrentYear']) * 100
        PEG_ratio = t.info['forwardPE'] / forward_EPS_growth

        # KPI #2: ROIC (ROIC = NOPAT / Invested Capital
        # 
        # --> NOPAT = EBIT * (1 - tax rate)
        # --> Invested Capital = Total Debt + Stockholders Equity - Cash and Cash Equivalents

        ebit = t.financials.loc['EBIT'].iloc[0]
        tax_Rate = t.financials.loc["Tax Rate For Calcs"].iloc[0]
        nopat = ebit * (1 - tax_Rate)

        tot_Debt = t.balance_sheet.loc["Total Debt"].iloc[0]
        tot_Equity = t.balance_sheet.loc["Stockholders Equity"].iloc[0]
        cash = t.balance_sheet.loc["Cash And Cash Equivalents"].iloc[0]
        invested_Capital = tot_Debt + tot_Equity - cash

        roic = (nopat / invested_Capital) * 100


        # I also get the company name
        name = t.info['shortName']


        # Add the KPIs to the list.
        kpi_data.append({
            "Ticker": ticker,
            "PEG Ratio (1Y)": round(PEG_ratio, 2),
            "ROIC (%)": round(roic, 2),
            "Name": name
        })

    except Exception as e:
        print(f"Could not get data for {ticker}: {e}")

Could not get data for LYC.F: 'forwardEps'
Could not get data for 002460.KS: 'forwardEps'
Could not get data for 3402.T: 'epsCurrentYear'
Could not get data for 4188.T: 'epsCurrentYear'
Could not get data for 002230.KQ: 'forwardEps'
Could not get data for 2454.T: 'forwardEps'
Could not get data for 373220.KS: 'forwardEps'
Could not get data for 006400.KS: 'forwardEps'
Could not get data for 6752.T: 'epsCurrentYear'
Could not get data for 6981.T: 'epsCurrentYear'
Could not get data for 6762.T: 'epsCurrentYear'
Could not get data for SONY: 'epsCurrentYear'
Could not get data for 005930.KS: 'forwardEps'
Could not get data for LAZR: 'forwardEps'
Could not get data for 2498.T: 'forwardEps'
Could not get data for 6324.T: 'epsCurrentYear'
Could not get data for 6268.T: 'epsCurrentYear'
Could not get data for 6481.T: 'epsCurrentYear'
Could not get data for 6594.T: 'epsCurrentYear'
Could not get data for 603728.SS: 'epsCurrentYear'
Could not get data for 002979.SZ: 'epsCurrentYear'
Could not ge

In [15]:
df_kpis = pd.DataFrame(kpi_data)
df_kpis

,Ticker,PEG Ratio (1Y),ROIC (%),Name
0,MP,0.02,-3.46,MP Materials Corp.
1,USAR,-4.56,-219.92,"USA Rare Earth, Inc."
2,UUUU,-0.08,-7.76,Energy Fuels Inc
3,ALB,-4.11,-2.43,Albemarle Corporation
4,SQM,-1.40,8.77,Sociedad Quimica y Minera S.A.
...,...,...,...,...
66,WMT,2.83,15.63,Walmart Inc.
67,MELI,0.66,14.58,"MercadoLibre, Inc."
68,PTC,1.77,15.92,PTC Inc.
69,NOW,1.14,15.17,"ServiceNow, Inc."


# Export

In [16]:
# Export
df_kpis.to_csv("../data/processed/stocks_kpis.csv", index=False)